In [1]:
from pathlib import Path
import re
import numpy as np
import matplotlib.pyplot as plt
import mcstasscript as ms
import csv

In [ ]:
# ============================================================
# User settings
# ============================================================
flatness_root = Path(
    '/Users/christian/Documents/Study/bachelor_project/geometry_optimization/cylinder_instrument/flatness_test'
)
nominal_detector_center_deg = 75.0
center_offsets_deg = np.arange(-30.0, 30.1, 2.0)
detector_window_width_deg = 60.0
low_outlier_fraction = 0.0
equator_band_halfwidth_local = 50.0
beam_lon_halfwidth_deg_local = 2.0
beam_lat_halfwidth_deg_local = 4.0

# ============================================================
# Helper functions
# ============================================================

def parse_mccode_sim(sim_file):
    params = {}
    pattern = re.compile(r'^\s*Param:\s*([A-Za-z0-9_]+)=(.+?)\s*$')
    with sim_file.open('r') as f:
        for line in f:
            match = pattern.match(line)
            if not match:
                continue
            key = match.group(1).strip()
            raw_value = match.group(2).strip()
            try:
                params[key] = float(raw_value)
            except ValueError:
                pass

    # safecguard against missing parameters
    required = ['cylinder_radius', 'cylinder_height', 'thickness']
    missing = [k for k in required if k not in params]
    if missing:
        raise ValueError(f'Missing {missing} in {sim_file}')

    # convert to mm and return
    return {
        'outer_radius_mm':
            params['cylinder_radius'] * 1000.0,
        'outer_height_mm':
            params['cylinder_height'] * 1000.0,
        'wall_thickness_mm':
            params['thickness'] * 1000.0,
    }

def load_scattered_monitor(run_dir):
    loaded = ms.load_data(str(run_dir))
    if not loaded:
        raise ValueError(
            f'No McStas data loaded from {run_dir}'
        )

    for item in loaded:
        name = str(getattr(item, 'name', '') or '')

        component = str(getattr(item, 'component_name', '') or '')

        if (
            'cylinder_scattered' in name or
            'mon_4pi' in component
        ):
            return item

    return loaded[2] if len(loaded) >= 3 else loaded[-1]


def get_corrected_heatmap_from_run(run_dir):
    sphere_mon = load_scattered_monitor(run_dir)
    meta = sphere_mon.metadata
    H = np.array(
        sphere_mon.Intensity,
        copy=True
    )

    H_err = np.array(
        sphere_mon.Error,
        copy=True
    )

    if (
        hasattr(meta, 'dimension') and
        H.shape == (
            meta.dimension[0],
            meta.dimension[1]
        )
    ):
        H = H.T
        H_err = H_err.T

    xmin, xmax, ymin, ymax = meta.limits

    lat_centers = np.linspace(
        ymin + 0.5 * (ymax - ymin) / H.shape[0],
        ymax - 0.5 * (ymax - ymin) / H.shape[0],
        H.shape[0],
    )

    lon_centers = np.linspace(
        xmin + 0.5 * (xmax - xmin) / H.shape[1],
        xmax - 0.5 * (xmax - xmin) / H.shape[1],
        H.shape[1],
    )

    cos_correction = np.cos(
        np.radians(lat_centers)
    )[:, np.newaxis]

    safe_cos = np.where(
        np.abs(cos_correction) > 0,
        cos_correction,
        np.nan
    )

    H_corr = H / safe_cos
    H_err_corr = H_err / safe_cos

    Lon, Lat = np.meshgrid(
        lon_centers,
        lat_centers
    )

    beam_center_mask = (
        (np.abs(Lon) <= beam_lon_halfwidth_deg_local)
        &
        (np.abs(Lat) <= beam_lat_halfwidth_deg_local)
    )

    H_corr = np.where(
        ~beam_center_mask,
        H_corr,
        np.nan
    )

    H_err_corr = np.where(
        ~beam_center_mask,
        H_err_corr,
        np.nan
    )

    return lon_centers, lat_centers, H_corr, H_err_corr


def get_equator_cut_from_run(run_dir):
    sphere_mon = load_scattered_monitor(run_dir)
    meta = sphere_mon.metadata
    Hmc = np.array(
        sphere_mon.Intensity,
        copy=True
    )

    Hmc_err = np.array(
        sphere_mon.Error,
        copy=True
    )

    if (
        hasattr(meta, 'dimension') and
        Hmc.shape == (
            meta.dimension[0],
            meta.dimension[1]
        )
    ):
        Hmc = Hmc.T
        Hmc_err = Hmc_err.T

    xmin, xmax, ymin, ymax = meta.limits

    lat_centers = np.linspace(
        ymin + 0.5 * (ymax - ymin) / Hmc.shape[0],
        ymax - 0.5 * (ymax - ymin) / Hmc.shape[0],
        Hmc.shape[0],
    )

    lon = np.linspace(xmin, xmax, Hmc.shape[1])

    cos_correction = np.cos(
        np.radians(lat_centers)
    )[:, np.newaxis]

    safe_cos = np.where(
        np.abs(cos_correction) > 0,
        cos_correction,
        np.nan
    )

    Hmc_corrected = Hmc / safe_cos
    Hmc_err_corrected = Hmc_err / safe_cos

    Lon, Lat = np.meshgrid(lon, lat_centers)

    beam_center_mask = (
        (np.abs(Lon) <= beam_lon_halfwidth_deg_local)
        &
        (np.abs(Lat) <= beam_lat_halfwidth_deg_local)
    )

    Hmc_corrected = np.where(
        ~beam_center_mask,
        Hmc_corrected,
        np.nan
    )

    Hmc_err_corrected = np.where(
        ~beam_center_mask,
        Hmc_err_corrected,
        np.nan
    )

    equator_band_mask = (
        np.abs(lat_centers)
        <= equator_band_halfwidth_local
    )

    equator_band = Hmc_corrected[
        equator_band_mask,
        :
    ]

    equator_band_err = Hmc_err_corrected[
        equator_band_mask,
        :
    ]

    nonzero_band = equator_band > 0

    equator_band = np.where(
        nonzero_band,
        equator_band,
        np.nan
    )

    equator_band_err = np.where(
        nonzero_band,
        equator_band_err,
        np.nan
    )

    equator_cut = np.nanmean(
        equator_band,
        axis=0
    )

    valid_counts = np.sum(
        ~np.isnan(equator_band),
        axis=0
    )

    equator_cut_err = np.where(
        valid_counts > 0,
        np.sqrt(
            np.nansum(equator_band_err**2, axis=0)
        ) / valid_counts,
        np.nan,
    )

    valid_cols = (
        (valid_counts > 0)
        &
        np.isfinite(equator_cut)
        &
        (equator_cut > 0)
    )

    low_outlier_threshold = (
        low_outlier_fraction
        *
        np.nanmedian(equator_cut[valid_cols])
    )

    valid_cols = (
        valid_cols
        &
        (equator_cut >= low_outlier_threshold)
    )

    return (
        lon,
        equator_cut,
        equator_cut_err,
        valid_cols
    )


def cylinder_volume(run_dir):
    params = parse_mccode_sim(run_dir / 'simulation.instr')
    r = params['outer_radius_mm']
    h = params['outer_height_mm']
    t = params['wall_thickness_mm']
    inner_r = r - t
    inner_h = h - 2 * t
    if inner_r <= 0 or inner_h <= 0:
        return 0.0
    return np.pi((r**2 * h) - (inner_r**2 * inner_h))

def flux_at_sample(run_dir):
    sphere_mon = load_scattered_monitor(run_dir)
    meta = sphere_mon.metadata
    Hmc = np.array(
        sphere_mon.Intensity,
        copy=True
    )

    Hmc_err = np.array(
        sphere_mon.Error,
        copy=True
    )

    if (
        hasattr(meta, 'dimension') and
        Hmc.shape == (
            meta.dimension[0],
            meta.dimension[1]
        )
    ):
        Hmc = Hmc.T
        Hmc_err = Hmc_err.T

    xmin, xmax, ymin, ymax = meta.limits

    lat_centers = np.linspace(
        ymin + 0.5 * (ymax - ymin) / Hmc.shape[0],
        ymax - 0.5 * (ymax - ymin) / Hmc.shape[0],
        Hmc.shape[0],
    )

    lon_centers = np.linspace(
        xmin + 0.5 * (xmax - xmin) / Hmc.shape[1],
        xmax - 0.5 * (xmax - xmin) / Hmc.shape[1],
        Hmc.shape[1],
    )

    cos_correction = np.cos(
        np.radians(lat_centers)
    )[:, np.newaxis]

    safe_cos = np.where(
        np.abs(cos_correction) > 0,
        cos_correction,
        np.nan
    )

    Hmc_corrected = Hmc / safe_cos
    Hmc_err_corrected = Hmc_err / safe_cos

    beam_center_mask = (
        (np.abs(lon_centers[:, np.newaxis]) <= beam_lon_halfwidth_deg_local)
        &
        (np.abs(lat_centers[np.newaxis, :]) <= beam_lat_halfwidth_deg_local)
    )

    total_flux = np.nansum(
        Hmc_corrected[beam_center_mask]
    )

    return total_flux


fhit_dir = Path('geometry_optimization/cylinder_instrument/1to3_r1cmh5cm_bathed_1e9/4')
if not fhit_dir.exists():
    fhit_dir = Path('cylinder_instrument/fhit_r5mmh5cm')

def load_mccode_2d(path):
    path = Path(path)
    meta = {}
    nrows = None
    data_rows = []
    in_data = False
    with path.open() as f:
        for line in f:
            if line.startswith('# type: array_2d'):
                dims = line.split('array_2d(', 1)[1].split(')', 1)[0]
                nrows = int(dims.split(',')[1].strip())
            elif line.startswith('# xylimits:'):
                meta['extent'] = [float(v) for v in line.split(':', 1)[1].split()]
            elif line.startswith('# component:'):
                meta['component'] = line.split(':', 1)[1].strip()
            elif line.startswith('# Data'):
                in_data = True
            elif in_data and line.startswith('#'):
                break
            elif in_data and line.strip():
                data_rows.append([float(v) for v in line.split()])
                if nrows is not None and len(data_rows) == nrows:
                    break

    if nrows is None:
        raise ValueError(f'Could not find array_2d dimensions in {path}')
    if len(data_rows) != nrows:
        raise ValueError(f'Expected {nrows} rows from {path}, got {len(data_rows)}')

    data = np.array(data_rows)
    return data, meta

before, before_meta = load_mccode_2d(fhit_dir / 'before.dat')
after, after_meta = load_mccode_2d(fhit_dir / 'after.dat')
extent = before_meta.get('extent', [-10, 10, -10, 10])

positive_values = np.concatenate([before[before > 0], after[after > 0]])
norm = LogNorm(vmin=positive_values.min(), vmax=positive_values.max())

fig, axs = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
for ax, image, title in [
    (axs[0], before, 'Before sample'),
    (axs[1], after, 'After sample'),
]:
    im = ax.imshow(
        np.ma.masked_less_equal(image, 0),
        extent=extent,
        origin='lower',
        aspect='equal',
        cmap='viridis',
        norm=norm,
    )
    ax.set_title(f'fhit {title}')
    ax.set_xlabel('X position [cm]')
    ax.set_ylabel('Y position [cm]')
    ax.axhline(0, color='white', lw=0.7, alpha=0.5)
    ax.axvline(0, color='white', lw=0.7, alpha=0.5)

fig.colorbar(im, ax=axs, label='Signal per bin (log scale)')
plt.show()

bin_width_x = (extent[1] - extent[0]) / before.shape[1]
bin_width_y = (extent[3] - extent[2]) / before.shape[0]
bin_area = abs(bin_width_x * bin_width_y)
beam_area_cm2 = np.count_nonzero(before > 0) * bin_area



    